In [1]:
import pandas as pd
import numpy as np
import selenium
import urllib.request as request
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import random
import regex as re
import undetected_chromedriver as uc
from selenium.webdriver.common.keys import Keys

In [2]:
df = pd.read_csv("filtered_movie_plots.csv")

In [3]:
df.shape


(997, 8)

In [5]:
df.dtypes

Release Year         int64
Title               object
Origin/Ethnicity    object
Director            object
Cast                object
Genre               object
Wiki Page           object
Plot                object
dtype: object

In [4]:
df.head()

,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot
0,2015,Temper,Telugu,Puri Jagannadh,"N. T. Rama Rao Jr., Kajal Aggarwal, Prakash Raj",action,https://en.wikipedia.org/wiki/Temper_(film),Daya is an orphan who grows up learning that a...
1,2009,Fighting,American,Dito Montiel,"Channing Tatum, Terrence Howard",action,https://en.wikipedia.org/wiki/Fighting_(2009_f...,Present day New York City: Shawn MacArthur (Ch...
2,2009,Newtonin Moondram Vidhi,Tamil,Thai Muthuselvam,"S. J. Surya, Sayali Bhagat, Rajiv Krishna\r\n",action,https://en.wikipedia.org/wiki/Newtonin_Moondra...,The film opens with a grim and bearded angry y...
3,2012,Dark Tide,American,John Stockwell,"Halle Berry, Olivier Martinez, Ralph Brown, Lu...",action,https://en.wikipedia.org/wiki/Dark_Tide,Kate is a shark expert whose business has been...
4,2011,Shakti,Telugu,Meher Ramesh,"Jr. NTR, Ileana D'Cruz, Manjari Phadnis, Jacki...",action,https://en.wikipedia.org/wiki/Shakti_(2011_film),Aishwarya (Ileana D'Cruz) is the daughter of c...


In [5]:
df = df.rename(columns={'Release Year': 'Release_Year', 'Wiki Page': 'Wiki_Page'})
df.head()

,Release_Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki_Page,Plot
0,2015,Temper,Telugu,Puri Jagannadh,"N. T. Rama Rao Jr., Kajal Aggarwal, Prakash Raj",action,https://en.wikipedia.org/wiki/Temper_(film),Daya is an orphan who grows up learning that a...
1,2009,Fighting,American,Dito Montiel,"Channing Tatum, Terrence Howard",action,https://en.wikipedia.org/wiki/Fighting_(2009_f...,Present day New York City: Shawn MacArthur (Ch...
2,2009,Newtonin Moondram Vidhi,Tamil,Thai Muthuselvam,"S. J. Surya, Sayali Bhagat, Rajiv Krishna\r\n",action,https://en.wikipedia.org/wiki/Newtonin_Moondra...,The film opens with a grim and bearded angry y...
3,2012,Dark Tide,American,John Stockwell,"Halle Berry, Olivier Martinez, Ralph Brown, Lu...",action,https://en.wikipedia.org/wiki/Dark_Tide,Kate is a shark expert whose business has been...
4,2011,Shakti,Telugu,Meher Ramesh,"Jr. NTR, Ileana D'Cruz, Manjari Phadnis, Jacki...",action,https://en.wikipedia.org/wiki/Shakti_(2011_film),Aishwarya (Ileana D'Cruz) is the daughter of c...


In [6]:
def get_movie_revenue_and_budget(driver, release_year,movie_name):
    url = 'https://www.themoviedb.org/'
    driver.get(url)
    time.sleep(random.uniform(1, 4)) 

   # search_box = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "search_v4")))
    formatted_name = movie_name.replace(" ", "%20")
    url = f"https://www.themoviedb.org/search/movie?language=en-US&query={formatted_name}%20y%3A{release_year}"
    driver.get(url)
    time.sleep(random.uniform(2, 4)) 
    # search_box.send_keys(movie_name + " " +"y:"+ str(release_year))
    # time.sleep(random.uniform(1, 4))
    # search_box.send_keys(Keys.ENTER)
    # movies_filter = WebDriverWait(driver, 3).until(
    #             EC.element_to_be_clickable((By.ID, "movie"))
    #         )
    # movies_filter.click()
    # time.sleep(random.uniform(1, 4))
    
    movie_link = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, '//a[@data-media-type="movie"]'))
    )
    
    # 2. Force a native JavaScript browser click (Completely bypasses rendering or coordinate bugs!)
    driver.execute_script("arguments[0].click();", movie_link)
    time.sleep(random.uniform(4, 7)) 
    try:
        revenue_element = driver.find_element(By.XPATH, "//p[strong/bdi[text()='Revenue']]")
        budget_element = driver.find_element(By.XPATH, "//p[strong/bdi[text()='Budget']]")

        revenue = revenue_element.text
        revenue_number = re.sub(r'\D', '', revenue)
        budget = budget_element.text
        budget_number = re.sub(r'\D', '', budget)
        time.sleep(random.uniform(1, 4)) 
    except Exception as e:
        revenue_number = None
        budget_number = None
    return revenue_number, budget_number



In [23]:
two_df = df[['Release_Year','Title']].head(2)
two_df.head()

,Release_Year,Title
0,2015,Temper
1,2009,Fighting


In [ ]:
driver = uc.Chrome(version_main=150, options=uc.ChromeOptions())
scraped_data_matrix = []

for row in two_df.itertuples(index=False):
    release_year, title = row
    try:
        revenue, budget = get_movie_revenue_and_budget(driver,release_year,title)
    except Exception as e:
        revenue, budget = None, None
    driver.refresh()
    time.sleep(5)  # Wait for 5 seconds before the next iteration
    scraped_data_matrix.append([revenue, budget])
    print(f"Scraped data for {title} ({release_year}): Revenue = {revenue}, Budget = {budget}")

driver.quit()
print("Scraping completed.")
print(scraped_data_matrix)

https://www.themoviedb.org/search/movie?language=en-US&query=Temper%20y%3A2015
Title: Temper, Revenue: 11600000, Budget: 5400000
Waiting for 5 seconds before the next iteration...
https://www.themoviedb.org/search/movie?language=en-US&query=Fighting%20y%3A2009
Title: Fighting, Revenue: 32474120, Budget: 
Waiting for 5 seconds before the next iteration...
Scraping completed.
[['11600000', '5400000'], ['32474120', '']]


In [27]:
revenue_list = [row[0] for row in scraped_data_matrix]
budget_list = [row[1] for row in scraped_data_matrix]

# 2. Assign them straight to the columns based purely on position order
two_df['Revenue'] = revenue_list
two_df['Budget'] = budget_list
two_df.head()

,Release_Year,Title,Revenue,Budget
0,2015,Temper,11600000,5400000
1,2009,Fighting,32474120,


In [8]:
mid = len(df) // 2

# Top half: 498 rows (Indices 0 to 497)
df_part1 = df.iloc[:mid].copy()

# Bottom half: 499 rows (Indices 498 to 996)
df_part2 = df.iloc[mid:].copy()

In [9]:
df_part1.shape, df_part2.shape

((498, 8), (499, 8))

In [34]:
two_df_part2.head(5)

,Release_Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki_Page,Plot
498,2008,Fireflies in the Garden,American,Dennis Lee,"Julia Roberts, Ryan Reynolds, Willem Dafoe, Em...",drama,https://en.wikipedia.org/wiki/Fireflies_in_the...,The story moves back and forth between the ado...
499,2007,Aha!,Bangladeshi,Enamul Kabir Nirjhor,"Humayun Faridi, Tariq Anam Khan, Fazlur Rahman...",drama,https://en.wikipedia.org/wiki/Aha!_(film),Mr. Mallik (Tariq Anam Khan) lives in an about...
500,2010,The Hangman,Bollywood,Vishal Bhandari,"Om Puri, Shreyas Talpade, Gulshan Grover, Smit...",drama,https://en.wikipedia.org/wiki/The_Hangman_(201...,"The Hangman, starring internationally acclaime..."
501,2014,Jamesy Boy,American,Trevor White,Spencer Lofranco\r\nMary-Louise Parker\r\nTais...,drama,https://en.wikipedia.org/wiki/Jamesy_Boy,"James Burns is in prison for selling guns, dru..."
502,2011,Funkytown,Canadian,Daniel Roby,"Patrick Huard, Raymond Bouchard, Geneviève Bro...",drama,https://en.wikipedia.org/wiki/Funkytown_(film),"Set in Montreal during the disco era, the film..."


In [42]:
df_part1['Title'] = df_part1['Title'].str.strip()

In [ ]:
driver = uc.Chrome(version_main=152, options=uc.ChromeOptions())
two_df_part1 = df_part1[['Release_Year','Title']]
scraped_data_matrix = []

for row in two_df_part1.itertuples(index=False):
    release_year, title = row
    try:
        revenue, budget = get_movie_revenue_and_budget(driver,release_year,title)
    except Exception as e:
        revenue, budget = None, None
    time.sleep(5)  # Wait for 5 seconds before the next iteration
    scraped_data_matrix.append([revenue, budget])
    print(f"Scraped data for {title} ({release_year}): Revenue = {revenue}, Budget = {budget}")

driver.quit()
print("Scraping completed.")

Scraped data for Temper (2015): Revenue = 11600000, Budget = 5400000
Scraped data for Fighting (2009): Revenue = 32474120, Budget = 
Scraped data for Newtonin Moondram Vidhi (2009): Revenue = , Budget = 
Scraped data for Dark Tide (2012): Revenue = 432274, Budget = 25000000
Scraped data for Shakti (2011): Revenue = , Budget = 
Scraped data for Machete (2010): Revenue = 45491656, Budget = 10500000
Scraped data for Police Police (2010): Revenue = , Budget = 
Scraped data for Tins (2007): Revenue = , Budget = 
Scraped data for Expendables 2, TheThe Expendables 2 (2012): Revenue = 314975955, Budget = 100000000
Scraped data for Gandedhe (2010): Revenue = None, Budget = None
Scraped data for Ballistic: Ecks vs. Sever (2002): Revenue = 19924033, Budget = 70000000
Scraped data for Runner (2013): Revenue = 62600000, Budget = 30000000
Scraped data for Shivamani (2003): Revenue = , Budget = 
Scraped data for From Paris with Love (2010): Revenue = 52800000, Budget = 52000000
Scraped data for Simha

In [ ]:
driver = uc.Chrome(version_main=152, options=uc.ChromeOptions())
revenue, budget = get_movie_revenue_and_budget(driver,2001,"The Fast and the Furious")
print(f"Scraped data for The Fast and the Furious (2001): Revenue = {revenue}, Budget = {budget}")

https://www.themoviedb.org/search/movie?language=en-US&query=The%20Fast%20and%20the%20Furious%20y%3A2001
Scraped data for The Fast and the Furious (2001): Revenue = 207283925, Budget = 38000000


In [40]:
string = " The Fast and the Furious"
print(string.strip())
print(string)

The Fast and the Furious
 The Fast and the Furious


In [51]:
revenue_list = [row[0] for row in scraped_data_matrix]
budget_list = [row[1] for row in scraped_data_matrix]

# 2. Assign them straight to the columns based purely on position order
two_df_part1['Revenue'] = revenue_list
two_df_part1['Budget'] = budget_list
two_df_part1.head()

C:\Users\Stephen Williams\AppData\Local\Temp\ipykernel_16816\2717512443.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  two_df_part1['Revenue'] = revenue_list
C:\Users\Stephen Williams\AppData\Local\Temp\ipykernel_16816\2717512443.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  two_df_part1['Budget'] = budget_list


,Release_Year,Title,Revenue,Budget
0,2015,Temper,11600000,5400000
1,2009,Fighting,32474120,
2,2009,Newtonin Moondram Vidhi,,
3,2012,Dark Tide,432274,25000000
4,2011,Shakti,,


In [53]:
revenue_list = [row[0] for row in scraped_data_matrix]
budget_list = [row[1] for row in scraped_data_matrix]

# 2. Assign them straight to the columns based purely on position order
df_part1['Revenue'] = revenue_list
df_part1['Budget'] = budget_list
df_part1.head()

,Release_Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki_Page,Plot,Revenue,Budget
0,2015,Temper,Telugu,Puri Jagannadh,"N. T. Rama Rao Jr., Kajal Aggarwal, Prakash Raj",action,https://en.wikipedia.org/wiki/Temper_(film),Daya is an orphan who grows up learning that a...,11600000,5400000
1,2009,Fighting,American,Dito Montiel,"Channing Tatum, Terrence Howard",action,https://en.wikipedia.org/wiki/Fighting_(2009_f...,Present day New York City: Shawn MacArthur (Ch...,32474120,
2,2009,Newtonin Moondram Vidhi,Tamil,Thai Muthuselvam,"S. J. Surya, Sayali Bhagat, Rajiv Krishna\r\n",action,https://en.wikipedia.org/wiki/Newtonin_Moondra...,The film opens with a grim and bearded angry y...,,
3,2012,Dark Tide,American,John Stockwell,"Halle Berry, Olivier Martinez, Ralph Brown, Lu...",action,https://en.wikipedia.org/wiki/Dark_Tide,Kate is a shark expert whose business has been...,432274,25000000
4,2011,Shakti,Telugu,Meher Ramesh,"Jr. NTR, Ileana D'Cruz, Manjari Phadnis, Jacki...",action,https://en.wikipedia.org/wiki/Shakti_(2011_film),Aishwarya (Ileana D'Cruz) is the daughter of c...,,


In [52]:
print(scraped_data_matrix)

[['11600000', '5400000'], ['32474120', ''], ['', ''], ['432274', '25000000'], ['', ''], ['45491656', '10500000'], ['', ''], ['', ''], ['314975955', '100000000'], [None, None], ['19924033', '70000000'], ['62600000', '30000000'], ['', ''], ['52800000', '52000000'], ['', ''], ['', ''], ['', ''], ['39881', ''], ['870325439', '30100000'], ['30800000', '45000000'], ['', ''], ['', ''], ['810243', ''], ['', ''], ['', ''], ['207283925', '38000000'], ['', ''], ['', ''], ['19233280', '25000000'], ['', ''], ['716934779', '200000000'], ['', ''], ['', ''], ['', ''], ['3704408', '3500000'], ['', ''], ['', ''], ['704709660', '340000000'], ['', ''], ['', ''], ['', '6500000'], ['', ''], ['', ''], ['', ''], ['', ''], ['57777106', '66000000'], ['378858340', '150000000'], [None, None], ['', ''], ['521918', '14000000'], ['', ''], ['', ''], ['', ''], ['6282446', '10000000'], ['', ''], ['', ''], ['', ''], ['', ''], [None, None], [None, None], ['26000000', '13000000'], ['', ''], ['50000000', '850000'], ['', ''

In [54]:
df_part1.to_csv("scraped_movie_data_part1.csv", index=False)

In [ ]:
test = pd.read_csv("scraped_movie_data_part1.csv")
test.head()


(498, 10)

In [58]:
test.shape

(498, 10)

In [10]:
driver = uc.Chrome(version_main=150, options=uc.ChromeOptions())
two_df_part2 = df_part2[['Release_Year','Title']]
scraped_data_matrix = []

for row in two_df_part2.itertuples(index=False):
    release_year, title = row
    try:
        revenue, budget = get_movie_revenue_and_budget(driver,release_year,title)
    except Exception as e:
        revenue, budget = None, None
    time.sleep(5)  # Wait for 5 seconds before the next iteration
    scraped_data_matrix.append([revenue, budget])
    print(f"Scraped data for {title} ({release_year}): Revenue = {revenue}, Budget = {budget}")

driver.quit()
print("Scraping completed.")


Scraped data for Fireflies in the Garden (2008): Revenue = 6692182, Budget = 8000000
Scraped data for Aha! (2007): Revenue = , Budget = 
Scraped data for The Hangman (2010): Revenue = , Budget = 
Scraped data for Jamesy Boy (2014): Revenue = , Budget = 
Scraped data for Funkytown (2011): Revenue = , Budget = 7300000
Scraped data for Faust (2011): Revenue = , Budget = 14400000
Scraped data for American Dreams in China (2013): Revenue = 88000000, Budget = 
Scraped data for Wah Taj (2014): Revenue = None, Budget = None
Scraped data for Life Show (2002): Revenue = , Budget = 
Scraped data for Chitrangada (2012): Revenue = , Budget = 
Scraped data for Get on Up (2014): Revenue = 33448971, Budget = 30000000
Scraped data for 32aam adhyayam 23aam vaakyam (2015): Revenue = , Budget = 
Scraped data for Changeling (2008): Revenue = 113400000, Budget = 55000000
Scraped data for Phir Kabhi (2009): Revenue = , Budget = 
Scraped data for Million Dollar Arm (2014): Revenue = 38307627, Budget = 2500000

In [11]:
revenue_list = [row[0] for row in scraped_data_matrix]
budget_list = [row[1] for row in scraped_data_matrix]

# 2. Assign them straight to the columns based purely on position order
df_part2['Revenue'] = revenue_list
df_part2['Budget'] = budget_list
df_part2.head()

,Release_Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki_Page,Plot,Revenue,Budget
498,2008,Fireflies in the Garden,American,Dennis Lee,"Julia Roberts, Ryan Reynolds, Willem Dafoe, Em...",drama,https://en.wikipedia.org/wiki/Fireflies_in_the...,The story moves back and forth between the ado...,6692182,8000000
499,2007,Aha!,Bangladeshi,Enamul Kabir Nirjhor,"Humayun Faridi, Tariq Anam Khan, Fazlur Rahman...",drama,https://en.wikipedia.org/wiki/Aha!_(film),Mr. Mallik (Tariq Anam Khan) lives in an about...,,
500,2010,The Hangman,Bollywood,Vishal Bhandari,"Om Puri, Shreyas Talpade, Gulshan Grover, Smit...",drama,https://en.wikipedia.org/wiki/The_Hangman_(201...,"The Hangman, starring internationally acclaime...",,
501,2014,Jamesy Boy,American,Trevor White,Spencer Lofranco\r\nMary-Louise Parker\r\nTais...,drama,https://en.wikipedia.org/wiki/Jamesy_Boy,"James Burns is in prison for selling guns, dru...",,
502,2011,Funkytown,Canadian,Daniel Roby,"Patrick Huard, Raymond Bouchard, Geneviève Bro...",drama,https://en.wikipedia.org/wiki/Funkytown_(film),"Set in Montreal during the disco era, the film...",,7300000


In [12]:
df_part2.to_csv("scraped_movie_data_part2.csv", index=False)

In [13]:
df_part2.shape

(499, 10)

In [2]:
df = pd.read_csv("wiki_movie_plots_deduped.csv")
df.shape

(34886, 8)

In [11]:
df_1980s = df[(df['Release Year'] >= 1990) & (df['Release Year'] < 2000)]
df_1980s.shape

(4468, 8)

In [12]:
driver = uc.Chrome(version_main=152, options=uc.ChromeOptions())
two_df_1980s = df_1980s[['Release Year','Title']]
scraped_data_matrix = []
count = 0
for row in two_df_1980s.itertuples(index=False):
    release_year, title = row
    try:
        revenue, budget = get_movie_revenue_and_budget(driver,release_year,title)
    except Exception as e:
        revenue, budget = None, None
    time.sleep(5)  # Wait for 5 seconds before the next iteration
    scraped_data_matrix.append([revenue, budget])
    print(f"Scraped data for {title} ({release_year}): Revenue = {revenue}, Budget = {budget}")
    count += 1
    if count == 1300:
        break
    print(f"Scraped {count} movies so far.")

driver.quit()
print("Scraping completed.")

Scraped data for The Adventures of Ford Fairlane (1990): Revenue = 20423389, Budget = 49000000
Scraped 1 movies so far.
Scraped data for After Dark, My Sweet (1990): Revenue = , Budget = 
Scraped 2 movies so far.
Scraped data for Air America (1990): Revenue = 57661269, Budget = 35000000
Scraped 3 movies so far.
Scraped data for Alice (1990): Revenue = 57200000, Budget = 3000000
Scraped 4 movies so far.
Scraped data for Almost an Angel (1990): Revenue = 6939946, Budget = 25000000
Scraped 5 movies so far.
Scraped data for The Ambulance (1990): Revenue = , Budget = 4000000
Scraped 6 movies so far.
Scraped data for American Ninja 4: The Annihilation (1990): Revenue = 358047, Budget = 
Scraped 7 movies so far.
Scraped data for Andre's Mother (1990): Revenue = , Budget = 
Scraped 8 movies so far.
Scraped data for Angel Town (1990): Revenue = 855810, Budget = 1800000
Scraped 9 movies so far.
Scraped data for Another 48 Hrs. (1990): Revenue = 153518974, Budget = 38000000
Scraped 10 movies so f

In [16]:
revenue_list = [row[0] for row in scraped_data_matrix]
budget_list = [row[1] for row in scraped_data_matrix]

# 2. Assign them straight to the columns based purely on position order
temp_df = pd.DataFrame({
    'revenue': revenue_list,
    'budget': budget_list
})
temp_df.shape

(1300, 2)

In [17]:
temp_df.to_csv("scraped_movie_data_1980s.csv", index=False)